# Lab3 - Train Classifier (English Only)

Classifier training setup:
- Train languages: ['en']
- Features: early fusion (XLM-R text + PySceneDetect keyframe stats + sensorial)
- Target: majority label from `labels_task3_1` (`YES`/`NO`)


In [1]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
import joblib


In [2]:
# Cluster configuration copied from run.sh / run_parallel.sh
SLURM_PARTITION = 'long'
SLURM_CPUS_PER_TASK = 8
SLURM_MEM = '32G'
SLURM_GRES_SEQUENTIAL = 'shard:6'  # run.sh
SLURM_GRES_PARALLEL = 'shard:4'    # run_parallel.sh
CONDA_ENV = 'RFA2526pt'
PYTORCH_CUDA_ALLOC_CONF = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = PYTORCH_CUDA_ALLOC_CONF

PROJECT_ROOT = Path('/home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi')
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DELIVERABLES_DIR = PROJECT_ROOT / 'entregables'
WEIGHTS_DIR = PROJECT_ROOT / 'weights'

DELIVERABLES_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

FUSION_PATH = ARTIFACTS_DIR / 'fusion_features.npz'
META_PATH = ARTIFACTS_DIR / 'sample_metadata.csv'

GROUP_ID = 'BeingChillingWeWillWin'
MODEL_TAG = 'XLMRSceneSensor_LR_EN'
TEST_CASE = 'EXIST2026'
TRAIN_LANGS = ['en']

print('Training langs:', TRAIN_LANGS)
print('Using artifacts:', FUSION_PATH)


Training langs: ['en']
Using artifacts: /home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi/artifacts/fusion_features.npz


In [3]:
if not FUSION_PATH.exists() or not META_PATH.exists():
    raise FileNotFoundError('Run 01_build_multimodal_features.ipynb first to generate artifacts.')

bundle = np.load(FUSION_PATH, allow_pickle=True)
X_all = bundle['X_fusion']
y_all = bundle['y']
langs_all = bundle['langs'].astype(str)
sample_ids_all = bundle['sample_ids'].astype(str)

meta = pd.read_csv(META_PATH)
meta['sample_id'] = meta['sample_id'].astype(str)

print('X_all shape:', X_all.shape)
print('Total samples:', len(sample_ids_all))
print('Language distribution:', dict(pd.Series(langs_all).value_counts()))


X_all shape: (2524, 884)
Total samples: 2524
Language distribution: {'es': np.int64(1524), 'en': np.int64(1000)}


In [4]:
train_mask = np.isin(langs_all, TRAIN_LANGS) & (y_all >= 0)

X_train_pool = X_all[train_mask]
y_train_pool = y_all[train_mask]

if len(np.unique(y_train_pool)) < 2:
    raise RuntimeError('Training subset has <2 classes. Check labels and language filter.')

print('Training subset size:', X_train_pool.shape[0])
print('Class distribution:', dict(pd.Series(y_train_pool).value_counts()))


Training subset size: 1000
Class distribution: {0: np.int64(539), 1: np.int64(461)}


In [5]:
# Internal validation for quick quality check.
X_tr, X_va, y_tr, y_va = train_test_split(
    X_train_pool,
    y_train_pool,
    test_size=0.2,
    random_state=42,
    stratify=y_train_pool,
)

clf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2500, class_weight='balanced', n_jobs=-1)),
])

clf.fit(X_tr, y_tr)
val_pred = clf.predict(X_va)

print('Validation macro-F1:', f1_score(y_va, val_pred, average='macro'))
print(classification_report(y_va, val_pred, digits=4))


Validation macro-F1: 0.6688077077478924
              precision    recall  f1-score   support

           0     0.7019    0.6759    0.6887       108
           1     0.6354    0.6630    0.6489        92

    accuracy                         0.6700       200
   macro avg     0.6687    0.6695    0.6688       200
weighted avg     0.6713    0.6700    0.6704       200



In [6]:
# Final fit on complete language-specific training subset.
final_clf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2500, class_weight='balanced', n_jobs=-1)),
])
final_clf.fit(X_train_pool, y_train_pool)

# Requirement: generate predictions for all samples (both languages).
pred_all = final_clf.predict(X_all)
label_map = {0: 'NO', 1: 'YES'}
pred_labels = [label_map[int(v)] for v in pred_all]

model_path = WEIGHTS_DIR / f'{GROUP_ID}_{MODEL_TAG}.joblib'
joblib.dump(final_clf, model_path)
print('Saved model:', model_path)


Saved model: /home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi/weights/BeingChillingWeWillWin_XLMRSceneSensor_LR_EN.joblib


In [7]:
output_rows = []
for sid, pred_label in zip(sample_ids_all, pred_labels):
    output_rows.append({
        'test_case': TEST_CASE,
        'id': str(sid),
        'value': pred_label,
    })

json_path = DELIVERABLES_DIR / f'{GROUP_ID}_{MODEL_TAG}.json'
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(output_rows, f, ensure_ascii=False)

print('Saved prediction json:', json_path)
print('Rows:', len(output_rows))
output_rows[:3]


Saved prediction json: /home/alumno.upv.es/scheng1/Master-IA-ALC/lab3/notebooks_shiyi/entregables/BeingChillingWeWillWin_XLMRSceneSensor_LR_EN.json
Rows: 2524


[{'test_case': 'EXIST2026', 'id': '120001', 'value': 'YES'},
 {'test_case': 'EXIST2026', 'id': '120002', 'value': 'NO'},
 {'test_case': 'EXIST2026', 'id': '120003', 'value': 'NO'}]

## Notes

- This notebook predicts over all available samples in the generated feature bundle.
- If the official test split is provided later, regenerate `fusion_features.npz` including test samples and rerun this notebook.
